In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_data
from shufflenet_v2 import ShuffleNetV2
from helper_utils import training_loop

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Comparison of Lightweight models

### 1. Parameter Count

In [3]:
def count_params(model):

    # 1. Trainable Parameters (requires_grad = True)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 2. Non-Trainable Parameters 
    # This includes frozen parameters AND buffers (like BN moving stats) to match TF
    frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    buffers = sum(b.numel() for b in model.buffers())
    non_trainable_total = frozen_params + buffers

    # 3. Total Parameters
    total_params = trainable_params + non_trainable_total

    print(f"{'Total Parameters:':<25} {total_params:,}")
    print(f"{'Trainable Parameters:':<25} {trainable_params:,}")
    print(f"{'Non-trainable Parameters:':<25} {non_trainable_total:,}")

In [4]:
shufflenetV2_standalone = ShuffleNetV2(n_class=10, model_size='1.0x')
fasternet_standalone = timm.create_model('fasternet_t0.in1k', pretrained=False, num_classes=10)
mobilenetV4_standalone = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=False, num_classes=10)

count_params(shufflenetV2_standalone)
count_params(fasternet_standalone)
count_params(mobilenetV4_standalone)

model size is  1.0x
Total Parameters:         1,281,236
Trainable Parameters:     1,265,000
Non-trainable Parameters: 16,236
Total Parameters:         2,647,007
Trainable Parameters:     2,637,310
Non-trainable Parameters: 9,697
Total Parameters:         2,530,968
Trainable Parameters:     2,505,834
Non-trainable Parameters: 25,134


### 2. Performance comparison between lightweight models

In [5]:
EPOCHS = 15
BATCH_SIZE = 32
LR = 0.001
NUM_CLASSES = 10

In [6]:
shufflenetV2_acc = []
fasternet_acc = []
mobilenetV4_acc = []

In [7]:
for run in range(5):
    print(f"Run {run + 1}/5")
    # Load data
    train_loader, val_loader = load_train_val_data(batch_size=BATCH_SIZE)
    
    # Train ShuffleNetV2 standalone
    shufflenetV2_standalone = ShuffleNetV2(n_class=NUM_CLASSES, model_size='1.0x')
    shufflenetV2_standalone_loss = nn.CrossEntropyLoss() 
    shufflenetV2_standalone_optimizer = optim.Adam(shufflenetV2_standalone.parameters(), lr=LR)
    _, shufflenet_standalone_history, = training_loop(
    model=shufflenetV2_standalone,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=shufflenetV2_standalone_loss,
    optimizer=shufflenetV2_standalone_optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=None,
    save_path=None,
    )
    shufflenetV2_acc.append(shufflenet_standalone_history['val_accuracy'][-1])
    
    
    # Train Fasternet standalone
    fasternet_standalone = timm.create_model('fasternet_t0.in1k', pretrained=False, num_classes=NUM_CLASSES)
    fasternet_standalone_loss = nn.CrossEntropyLoss()
    fasternet_standalone_optimizer = optim.Adam(fasternet_standalone.parameters(), lr=LR)
    _, fasternet_standalone_history = training_loop(
    model=fasternet_standalone,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=fasternet_standalone_loss,
    optimizer=fasternet_standalone_optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=None,
    save_path=None,
    )
    fasternet_acc.append(fasternet_standalone_history['val_accuracy'][-1])

    # train mobilenetV4 standalone
    mobilenetV4_standalone = timm.create_model('mobilenetv4_conv_small.e2400_r224_in1k', pretrained=False, num_classes=NUM_CLASSES)
    mobilenetV4_standalone_loss = nn.CrossEntropyLoss()
    mobilenetV4_standalone_optimizer = optim.Adam(mobilenetV4_standalone.parameters(), lr=LR)
    _, mobilenetV4_standalone_history = training_loop(
    model=mobilenetV4_standalone,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_function=mobilenetV4_standalone_loss,
    optimizer=mobilenetV4_standalone_optimizer,
    num_epochs=EPOCHS,
    device=device,
    scheduler=None,
    save_path=None,
    )
    mobilenetV4_acc.append(mobilenetV4_standalone_history['val_accuracy'][-1])

Run 1/5
model size is  1.0x


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 31.3449%, Train Loss: 1.9459, Val Loss: 1.6314, Val Acc: 44.5444%
Epoch 2/15 - Train Acc: 49.3914%, Train Loss: 1.4839, Val Loss: 1.3460, Val Acc: 52.9976%
Epoch 3/15 - Train Acc: 60.4808%, Train Loss: 1.1826, Val Loss: 1.3830, Val Acc: 56.3549%
Epoch 4/15 - Train Acc: 68.1593%, Train Loss: 0.9627, Val Loss: 1.0705, Val Acc: 65.3477%
Epoch 5/15 - Train Acc: 75.0263%, Train Loss: 0.7601, Val Loss: 1.6041, Val Acc: 57.5540%
Epoch 6/15 - Train Acc: 81.6679%, Train Loss: 0.5656, Val Loss: 0.8517, Val Acc: 73.2614%
Epoch 7/15 - Train Acc: 86.1758%, Train Loss: 0.4321, Val Loss: 0.8735, Val Acc: 74.1607%
Epoch 8/15 - Train Acc: 89.8272%, Train Loss: 0.3145, Val Loss: 0.9722, Val Acc: 72.1223%
Epoch 9/15 - Train Acc: 92.0210%, Train Loss: 0.2485, Val Loss: 0.9925, Val Acc: 73.6211%
Epoch 10/15 - Train Acc: 92.6521%, Train Loss: 0.2311, Val Loss: 0.7974, Val Acc: 77.9976%
Epoch 11/15 - Train Acc: 92.9677%, Train Loss: 0.2133, Val Loss: 0.7543, Val Acc: 79.6763%
Epoch 12

Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 27.2126%, Train Loss: 2.0299, Val Loss: 2.1864, Val Acc: 22.7218%
Epoch 2/15 - Train Acc: 31.3899%, Train Loss: 1.9195, Val Loss: 2.5131, Val Acc: 26.1391%
Epoch 3/15 - Train Acc: 38.6026%, Train Loss: 1.7366, Val Loss: 1.7314, Val Acc: 41.8465%
Epoch 4/15 - Train Acc: 41.9083%, Train Loss: 1.6317, Val Loss: 1.5912, Val Acc: 43.4053%
Epoch 5/15 - Train Acc: 46.0556%, Train Loss: 1.5202, Val Loss: 1.7334, Val Acc: 40.3477%
Epoch 6/15 - Train Acc: 48.2645%, Train Loss: 1.4679, Val Loss: 1.4379, Val Acc: 48.8609%
Epoch 7/15 - Train Acc: 51.1946%, Train Loss: 1.3910, Val Loss: 1.4479, Val Acc: 51.1990%
Epoch 8/15 - Train Acc: 54.6957%, Train Loss: 1.3254, Val Loss: 1.5665, Val Acc: 45.8034%
Epoch 9/15 - Train Acc: 55.5973%, Train Loss: 1.2533, Val Loss: 1.9519, Val Acc: 40.8273%
Epoch 10/15 - Train Acc: 58.1067%, Train Loss: 1.1820, Val Loss: 1.4277, Val Acc: 50.8393%
Epoch 11/15 - Train Acc: 60.0301%, Train Loss: 1.1396, Val Loss: 1.1309, Val Acc: 61.0312%
Epoch 12

...

Epoch 1/15 - Train Acc: 33.7190%, Train Loss: 1.8862, Val Loss: 1.6478, Val Acc: 43.2854%
Epoch 2/15 - Train Acc: 51.0293%, Train Loss: 1.4306, Val Loss: 1.5150, Val Acc: 50.0600%
Epoch 3/15 - Train Acc: 60.6612%, Train Loss: 1.1683, Val Loss: 1.2020, Val Acc: 57.8537%
Epoch 4/15 - Train Acc: 68.9557%, Train Loss: 0.9248, Val Loss: 1.0903, Val Acc: 64.6882%
Epoch 5/15 - Train Acc: 76.1082%, Train Loss: 0.7306, Val Loss: 0.9951, Val Acc: 69.6043%
Epoch 6/15 - Train Acc: 81.4575%, Train Loss: 0.5555, Val Loss: 0.8588, Val Acc: 72.4820%
Epoch 7/15 - Train Acc: 86.2960%, Train Loss: 0.4218, Val Loss: 0.9729, Val Acc: 71.7026%
Epoch 8/15 - Train Acc: 88.4448%, Train Loss: 0.3488, Val Loss: 0.8588, Val Acc: 73.6211%
Epoch 9/15 - Train Acc: 91.0443%, Train Loss: 0.2699, Val Loss: 0.6929, Val Acc: 78.2974%
Epoch 10/15 - Train Acc: 93.2832%, Train Loss: 0.2017, Val Loss: 0.8168, Val Acc: 77.0384%
Epoch 11/15 - Train Acc: 94.5304%, Train Loss: 0.1677, Val Loss: 0.7570, Val Acc: 79.2566%
Epoch 12

Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 27.3929%, Train Loss: 2.0411, Val Loss: 2.0061, Val Acc: 27.9377%
Epoch 2/15 - Train Acc: 32.6521%, Train Loss: 1.8947, Val Loss: 2.1957, Val Acc: 23.9808%
Epoch 3/15 - Train Acc: 37.5207%, Train Loss: 1.7722, Val Loss: 2.0274, Val Acc: 36.8106%
Epoch 4/15 - Train Acc: 43.4110%, Train Loss: 1.6360, Val Loss: 1.5808, Val Acc: 44.9041%
Epoch 5/15 - Train Acc: 46.9722%, Train Loss: 1.5190, Val Loss: 1.6776, Val Acc: 42.7458%
Epoch 6/15 - Train Acc: 50.0977%, Train Loss: 1.4384, Val Loss: 1.4837, Val Acc: 47.3621%
Epoch 7/15 - Train Acc: 52.2915%, Train Loss: 1.3495, Val Loss: 1.2711, Val Acc: 54.6163%
Epoch 8/15 - Train Acc: 55.8377%, Train Loss: 1.3004, Val Loss: 1.3709, Val Acc: 52.7578%
Epoch 9/15 - Train Acc: 57.6860%, Train Loss: 1.2115, Val Loss: 1.2914, Val Acc: 56.8345%
Epoch 10/15 - Train Acc: 61.0218%, Train Loss: 1.1367, Val Loss: 1.0475, Val Acc: 64.9281%
Epoch 11/15 - Train Acc: 63.3959%, Train Loss: 1.0651, Val Loss: 1.2398, Val Acc: 58.8729%
Epoch 12

Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 28.1593%, Train Loss: 2.5016, Val Loss: 2.1199, Val Acc: 32.2542%
Epoch 2/15 - Train Acc: 41.3073%, Train Loss: 1.7861, Val Loss: 1.6914, Val Acc: 44.1247%
Epoch 3/15 - Train Acc: 51.3449%, Train Loss: 1.5184, Val Loss: 1.6748, Val Acc: 50.6595%
Epoch 4/15 - Train Acc: 59.5793%, Train Loss: 1.2522, Val Loss: 1.5272, Val Acc: 53.2974%
Epoch 5/15 - Train Acc: 68.1142%, Train Loss: 1.0211, Val Loss: 1.5049, Val Acc: 58.0935%
Epoch 6/15 - Train Acc: 72.6672%, Train Loss: 0.8289, Val Loss: 1.3584, Val Acc: 60.1319%
Epoch 7/15 - Train Acc: 78.0165%, Train Loss: 0.7035, Val Loss: 1.2635, Val Acc: 61.5108%
Epoch 8/15 - Train Acc: 81.9835%, Train Loss: 0.5601, Val Loss: 1.3023, Val Acc: 62.4101%
Epoch 9/15 - Train Acc: 86.3561%, Train Loss: 0.4254, Val Loss: 1.0843, Val Acc: 71.6427%
Epoch 10/15 - Train Acc: 88.6401%, Train Loss: 0.3518, Val Loss: 1.5420, Val Acc: 64.5683%
Epoch 11/15 - Train Acc: 89.3614%, Train Loss: 0.3312, Val Loss: 0.9931, Val Acc: 73.5012%
Epoch 12

In [8]:
print(shufflenetV2_acc)
print(fasternet_acc)
print(mobilenetV4_acc)

[0.7823740839958191, 0.7673860788345337, 0.798561155796051, 0.776378870010376, 0.8093525171279907]
[0.5659472346305847, 0.7661870718002319, 0.6900479793548584, 0.6061151027679443, 0.6924460530281067]
[0.6822541952133179, 0.7523980736732483, 0.6618704795837402, 0.7248201370239258, 0.7559952139854431]


In [9]:
np.mean(shufflenetV2_acc),  np.mean(fasternet_acc), np.mean(mobilenetV4_acc)

(np.float64(0.7868105411529541),
 np.float64(0.6641486883163452),
 np.float64(0.7154676198959351))